# Housing Data Analysis Notebook
## Created by: Central Maine Growth Council
## Data Source: [Realtor.com](https://www.realtor.com/research/data/) Research Data

This notebook analyzes housing market metrics for a specified ZIP code, comparing against state and national trends. It produces:
- Seasonal adjustment analysis using Prophet
- Geographic comparison plots (ZIP vs State vs National)
- Yearly summary tables

### Requirements
1. This is a python3 based jupyter-notebook and requires that python is install along with jupyter-lab or note book
   - see this link to understand how to install these
2. The following python libraries are needed
   - pandas
   - numpy
   - math
   - os
   - warnings
   - prophet
   - plotly
   - kaleido
3. You must have google chrome installed

### Usage
1. Edit the **Configuration** cell below with your settings
2. Run all cells (Cell → Run All)
3. Check the outputs in the specified directories

## Section 1: Configuration

In [35]:
# =============================================================================
# USER CONFIGURATION - Edit these values
# =============================================================================

# Required: Geographic settings
ZIP_CODE = '04901'
STATE_CODE = 'ME'

# Required: Input file paths
ZIP_DATA_PATH = 'RDC_Inventory_Core_Metrics_Zip_History.csv'
STATE_DATA_PATH = 'RDC_Inventory_Core_Metrics_State_History.csv'
NATIONAL_DATA_PATH = 'RDC_Inventory_Core_Metrics_Country_History.csv'

# Required: Date fields. American date format: yyyy-mm-dd
from_date = '2015-01-01'
to_date = '2025-12-31'

# Required: Output directories
GRAPHS_OUTPUT_DIR = 'outputs/graphs'
TABLES_OUTPUT_DIR = 'outputs/tables'

# Analysis options
# INCLUDE_STATE_COMPARISON = True      # Compare ZIP to state
# INCLUDE_NATIONAL_COMPARISON = True   # Include national data in comparisons
# APPLY_SEASONAL_ADJUSTMENT = True     # Use Prophet for seasonal adjustment

# # Metrics to analyze (comment out any you don't need)
# METRICS = [
#     'active_listing_count',
#     'median_listing_price',
#     'average_listing_price',
#     'new_listing_count',
#     'median_days_on_market',
#     'median_square_feet',
#     'median_listing_price_per_square_foot',
# ]

In [36]:
# =============================================================================
# METRIC CONFIGURATION
# Each metric can have its own settings for:
#   - label: Display name for plots/tables
#   - log_compare: Use log scale for geographic comparisons
#   - seasonal: Apply seasonal adjustment
#   - state: Include state comparison
#   - national: Include national comparison
#   - agg: Aggregation for yearly summary ('sum' or 'mean')
# =============================================================================

METRIC_CONFIG = {
    "active_listing_count": {
        "label": "Active Listing Count",
        "log_compare": True,      # Log scale for ZIP vs State/National
        "seasonal": True,
        "state": True,
        "national": False,        # National comparison not meaningful
        "agg": "sum",
    },
    "median_listing_price": {
        "label": "Median Listing Price ($)",
        "log_compare": False,
        "seasonal": True,
        "state": True,
        "national": True,
        "agg": "mean",
    },
    "average_listing_price": {
        "label": "Average Listing Price ($)",
        "log_compare": False,
        "seasonal": True,
        "state": True,
        "national": True,
        "agg": "mean",
    },
    "new_listing_count": {
        "label": "New Listing Count",
        "log_compare": True,
        "seasonal": True,
        "state": True,
        "national": False,
        "agg": "sum",
    },
    "median_days_on_market": {
        "label": "Median Days on Market",
        "log_compare": False,
        "seasonal": True,
        "state": True,
        "national": True,
        "agg": "mean",
    },
    "median_square_feet": {
        "label": "Median Square Feet",
        "log_compare": False,
        "seasonal": False,        # Less seasonal variation
        "state": True,
        "national": True,
        "agg": "mean",
    },
    "median_listing_price_per_square_foot": {
        "label": "Median Price per Sq Ft ($)",
        "log_compare": False,
        "seasonal": True,
        "state": True,
        "national": True,
        "agg": "mean",
    },
}

In [37]:
# Metrics to analyze (comment out any you don't need)                                                                                                                                       
METRICS = list(METRIC_CONFIG.keys())                                                                                                                                                        
# Or pick specific ones by removing the pound sign below and adding the variables (with quotations marks around) of interest within the square brackets                                                                                                                                                                     
# METRICS = ['active_listing_count', 'median_listing_price']                            

## Section 2: Imports & Setup

In [38]:
import pandas as pd
import numpy as np
import math
import os
import warnings
from prophet import Prophet
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

# Suppress Prophet and pandas warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module='prophet')
warnings.filterwarnings('ignore', module='cmdstanpy')

# Check for kaleido (required for PNG export)
try:
    import kaleido
    KALEIDO_AVAILABLE = True
except ImportError:
    KALEIDO_AVAILABLE = False
    print("WARNING: kaleido not installed. PNG export will be disabled and the graphs printed in this notebook will not be comprehensive.")
    print("To enable: pip install kaleido")

# Set plotly renderer
pio.renderers.default = 'iframe_connected'

# Create output directories
os.makedirs(GRAPHS_OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLES_OUTPUT_DIR, exist_ok=True)

print(f"Output directories created:")
print(f"  Graphs: {GRAPHS_OUTPUT_DIR}")
print(f"  Tables: {TABLES_OUTPUT_DIR}")

Output directories created:
  Graphs: outputs/graphs
  Tables: outputs/tables


## Section 3: Helper Functions

### Data Loading Functions

In [39]:
def load_zip_data(path, zip_code):
    """Load and filter ZIP-level data, return with date index."""
    df = pd.read_csv(path, dtype={'postal_code': str})
    df = df.loc[df['postal_code'] == zip_code].copy()
    df['date'] = pd.to_datetime(df['month_date_yyyymm'].astype(str), format='%Y%m')
    df = df.sort_values('date', ascending=True).set_index('date')
    return df


def load_state_data(path, state_code):
    """Load and filter state-level data, return with date index."""
    df = pd.read_csv(path)
    df = df.loc[df['state_id'] == state_code].copy()
    df['date'] = pd.to_datetime(df['month_date_yyyymm'].astype(str), format='%Y%m')
    df = df.sort_values('date', ascending=True).set_index('date')
    return df


def load_national_data(path):
    """Load national data, return with date index."""
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['month_date_yyyymm'].astype(str), format='%Y%m')
    df = df.sort_values('date', ascending=True).set_index('date')
    return df


def get_town_name(df_zip):
    """Extract town name from ZIP data."""
    return df_zip["zip_name"].unique()[0].split(',')[0]

### Seasonal Adjustment Function

In [40]:
def seasonal_adjust(series):
    """
    Apply Prophet seasonal adjustment to a time series.
    
    Args:
        series: pandas Series with datetime index
        
    Returns:
        DataFrame with columns: ds (date), y (observed), yhat (seasonally adjusted)
    """
    dfp = series.copy()
    dfp.index.name = 'ds'
    dfp.name = 'y'
    dfp = dfp.reset_index()
    pd.DataFrame(dfp).to_csv('test.csv')
    
    # Fit Prophet model with multiplicative seasonality
    model = Prophet(seasonality_mode='multiplicative')
    model.fit(dfp)
    
    # Get predictions (no future periods, just fitted values)
    forecast = model.predict(model.make_future_dataframe(periods=0))
    
    # Combine observed and adjusted
    result = pd.DataFrame({
        'ds': dfp['ds'],
        'y': dfp['y'].values,
        'yhat': forecast['yhat'].values
    })
    
    return result

### Plotting Functions

In [41]:
CUSTOM_COLORS = ["#07336e", "#ffd500", "#0d6c9e", "#1c4e77"]


def plot_observed_vs_adjusted(dates, observed, adjusted, metric_label, location_name, zip_code):
    """
    Plot observed vs seasonally adjusted data for a single location.
    
    Returns:
        plotly Figure object
    """
    fig = go.Figure()
    
    # Observed line (solid)
    fig.add_trace(go.Scatter(
        x=dates,
        y=observed,
        mode="lines",
        name="Observed",
        line=dict(color=CUSTOM_COLORS[0], width=3, dash="solid")
    ))
    
    # Seasonally adjusted line (dashed)
    fig.add_trace(go.Scatter(
        x=dates,
        y=adjusted,
        mode="lines",
        name="Seasonally Adjusted",
        line=dict(color=CUSTOM_COLORS[0], width=3, dash="dash")
    ))
    
    # End annotations
    obs_val = round(observed.iloc[-1] if hasattr(observed, "iloc") else observed[-1])
    adj_val = round(adjusted.iloc[-1] if hasattr(adjusted, "iloc") else adjusted[-1])
    
    fig.add_annotation(
        x=dates.iloc[-1], y=observed.iloc[-1],
        text=f"{obs_val:,}", showarrow=False,
        font=dict(color=CUSTOM_COLORS[0], size=12),
        yshift=10, bgcolor="white", borderpad=2
    )
    fig.add_annotation(
        x=dates.iloc[-1], y=adjusted.iloc[-1],
        text=f"{adj_val:,}", showarrow=False,
        font=dict(color=CUSTOM_COLORS[0], size=12),
        yshift=-10, bgcolor="white", borderpad=2
    )
    
    fig.update_layout(
        template="plotly_white",
        width=1000, height=600,
        title=f"{metric_label} | {location_name} ({zip_code}) | Observed vs Seasonally Adjusted",
        xaxis_title="Date",
        yaxis_title=metric_label,
        legend=dict(
            x=0.01, y=0.99, xanchor="left",
            bgcolor="rgba(255,255,255,0.75)",
            bordercolor="lightgray", borderwidth=1
        ),
        margin=dict(l=60, r=20, t=60, b=50)
    )
    fig.update_xaxes(type='date')
    
    return fig


def plot_zip_vs_state(zip_dates, zip_values, state_dates, state_values,
                      metric_label, zip_name, state_name, zip_code):
    """
    Plot ZIP vs State comparison.
    
    Returns:
        plotly Figure object
    """
    fig = go.Figure()
    
    # ZIP series
    fig.add_trace(go.Scatter(
        x=zip_dates, y=zip_values,
        mode="lines", name=f"{zip_name} ({zip_code})",
        line=dict(color=CUSTOM_COLORS[0], width=3)
    ))
    
    # State series
    fig.add_trace(go.Scatter(
        x=state_dates, y=state_values,
        mode="lines", name=state_name,
        line=dict(color=CUSTOM_COLORS[1], width=3)
    ))
    
    # End annotations
    zip_val = round(zip_values.iloc[-1])
    state_val = round(state_values.iloc[-1])
    
    offset = 10 if zip_val >= state_val else -10
    fig.add_annotation(
        x=zip_dates.iloc[-1], y=zip_values.iloc[-1],
        text=f"{zip_val:,}", showarrow=False,
        font=dict(color=CUSTOM_COLORS[0], size=12),
        yshift=offset, bgcolor="white", borderpad=2
    )
    fig.add_annotation(
        x=state_dates.iloc[-1], y=state_values.iloc[-1],
        text=f"{state_val:,}", showarrow=False,
        font=dict(color=CUSTOM_COLORS[1], size=12),
        yshift=-offset, bgcolor="white", borderpad=2
    )
    
    fig.update_layout(
        template="plotly_white",
        width=1000, height=600,
        title=f"{metric_label} | {zip_name} ({zip_code}) vs {state_name}",
        xaxis_title="Date",
        yaxis_title=metric_label,
        legend=dict(
            x=0.01, y=0.99, xanchor="left",
            bgcolor="rgba(255,255,255,0.75)",
            bordercolor="lightgray", borderwidth=1
        ),
        margin=dict(l=60, r=20, t=60, b=50)
    )
    fig.update_xaxes(type='date')
    
    return fig


def plot_three_geographies(zip_dates, zip_values, state_dates, state_values,
                           nat_dates, nat_values, metric_label, zip_name, state_name, zip_code):
    """
    Plot ZIP/State/National comparison.
    
    Returns:
        plotly Figure object
    """
    fig = go.Figure()
    
    # ZIP series
    fig.add_trace(go.Scatter(
        x=zip_dates, y=zip_values,
        mode="lines", name=f"{zip_name} ({zip_code})",
        line=dict(color=CUSTOM_COLORS[0], width=3)
    ))
    
    # State series
    fig.add_trace(go.Scatter(
        x=state_dates, y=state_values,
        mode="lines", name=state_name,
        line=dict(color=CUSTOM_COLORS[1], width=3)
    ))
    
    # National series
    fig.add_trace(go.Scatter(
        x=nat_dates, y=nat_values,
        mode="lines", name="US National",
        line=dict(color=CUSTOM_COLORS[2], width=3)
    ))
    
    # End annotations with offset adjustment
    vals = [
        (zip_values.iloc[-1], CUSTOM_COLORS[0], zip_dates.iloc[-1]),
        (state_values.iloc[-1], CUSTOM_COLORS[1], state_dates.iloc[-1]),
        (nat_values.iloc[-1], CUSTOM_COLORS[2], nat_dates.iloc[-1])
    ]
    sorted_vals = sorted(vals, key=lambda x: x[0])
    offsets = [-15, 0, 15]
    
    for (val, color, date), offset in zip(sorted_vals, offsets):
        fig.add_annotation(
            x=date, y=val,
            text=f"{round(val):,}", showarrow=False,
            font=dict(color=color, size=12),
            yshift=offset, bgcolor="white", borderpad=2
        )
    
    fig.update_layout(
        template="plotly_white",
        width=1000, height=600,
        title=f"{metric_label} | {zip_name} ({zip_code}) vs {state_name} vs US National",
        xaxis_title="Date",
        yaxis_title=metric_label,
        legend=dict(
            x=0.01, y=0.99, xanchor="left",
            bgcolor="rgba(255,255,255,0.75)",
            bordercolor="lightgray", borderwidth=1
        ),
        margin=dict(l=60, r=20, t=60, b=50)
    )
    fig.update_xaxes(type='date')
    
    return fig


def save_plot(fig, filepath):
    """Save figure as PNG if kaleido is available."""
    if KALEIDO_AVAILABLE:
        fig.write_image(filepath, scale=2)
        print(f"  Saved: {filepath}")
    else:
        print(f"  Skipped PNG save (kaleido not installed): {filepath}")

### Table Functions

In [42]:
def create_yearly_summary(data_dict, metric_name, agg_func='mean'):
    """
    Create yearly summary table from multiple series.
    
    Args:
        data_dict: dict of {label: (dates, values)} pairs
        metric_name: name of the metric (for determining aggregation)
        agg_func: 'mean' or 'sum' (defaults based on metric type)
    
    Returns:
        DataFrame with years as columns, series labels as rows
    """
    # Determine aggregation function based on metric
    count_metrics = ['active_listing_count', 'new_listing_count']
    if metric_name in count_metrics:
        agg_func = 'sum'
    else:
        agg_func = 'mean'
    
    # Build combined dataframe
    dfs = []
    for label, (dates, values) in data_dict.items():
        df = pd.DataFrame({'Date': dates, label: values})
        df['Year'] = pd.to_datetime(df['Date']).dt.year
        dfs.append(df[['Year', label]])
    
    # Merge all series
    combined = dfs[0]
    for df in dfs[1:]:
        combined = combined.merge(df, on='Year', how='outer')
    
    # Aggregate by year
    if agg_func == 'sum':
        result = combined.groupby('Year').sum().round(0)
    else:
        result = combined.groupby('Year').mean().round(1)
    
    return result.T


def save_table(df, filepath):
    """Save DataFrame as CSV."""
    df.to_csv(filepath)
    print(f"  Saved: {filepath}")

### Main Analysis Function

In [43]:
def analyze_metric(metric_name, metric_label, df_zip, df_state, df_national,
                   town_name, state_code, config):
    """
    Complete analysis pipeline for a single metric:
    1. Apply seasonal adjustment (if enabled)
    2. Generate all applicable plots
    3. Create and save yearly summary table
    4. Save all plots as PNG
    
    Args:
        metric_name: column name in dataframes
        metric_label: display name for plots/tables
        df_zip: ZIP-level dataframe
        df_state: State-level dataframe
        df_national: National dataframe
        town_name: name of town for labels
        state_code: 2-letter state code
        config: dict with analysis options
    """
    zip_code = ZIP_CODE
    graphs_dir = GRAPHS_OUTPUT_DIR
    tables_dir = TABLES_OUTPUT_DIR
    
    display(Markdown(f"### {metric_label}"))
    
    # Get raw data series
    zip_series = df_zip[metric_name].dropna()
    state_series = df_state[metric_name].dropna()
    nat_series = df_national[metric_name].dropna()

    # if config asks for log comparison, for things like active listing, perform transformation on the whole series
    if config['log_compare']:
        zip_series = np.log1p(zip_series)
        state_series = np.log1p(state_series)
        nat_series = np.log1p(nat_series)
    
    # Track data for yearly summary
    table_data = {}
    
    # --- Seasonal adjustment (if enabled) ---
    if config['seasonal']:
        print("  Applying seasonal adjustment...")
        adjusted = seasonal_adjust(zip_series)
        
        # Plot 1: Observed vs Seasonally Adjusted
        fig = plot_observed_vs_adjusted(
            adjusted['ds'], adjusted['y'], adjusted['yhat'],
            metric_label, town_name, zip_code
        )
        # fig.show()
        save_plot(fig, f"{graphs_dir}/{zip_code}_{metric_name}_seasonal.png")
        
        # Store for table
        table_data[f"{town_name} (Observed)"] = (adjusted['ds'], adjusted['y'])
        table_data[f"{town_name} (Seasonally Adjusted)"] = (adjusted['ds'], adjusted['yhat'])
        
        # Use adjusted values for comparisons
        zip_dates = adjusted['ds']
        zip_values = adjusted['yhat']
    else:
        # Use raw values
        zip_dates = zip_series.index.to_series().reset_index(drop=True)
        zip_values = zip_series.reset_index(drop=True)
        table_data[f"{town_name} (Observed)"] = (zip_dates, zip_values)
    
    # Prepare state data
    state_dates = state_series.index.to_series().reset_index(drop=True)
    state_values = state_series.reset_index(drop=True)
    
    # --- State comparison plot ---
    if config['state']:
        state_name = state_code  # Could map to full name
        
        fig = plot_zip_vs_state(
            zip_dates, zip_values,
            state_dates, state_values,
            metric_label, town_name, state_name, zip_code
        )
        # fig.show()
        save_plot(fig, f"{graphs_dir}/{zip_code}_{metric_name}_vs_{state_code}.png")
        
        table_data[f"{state_name} (Observed)"] = (state_dates, state_values)
    
    # --- Three geography comparison plot ---
    if config['national']:
        nat_dates = nat_series.index.to_series().reset_index(drop=True)
        nat_values = nat_series.reset_index(drop=True)
        
        fig = plot_three_geographies(
            zip_dates, zip_values,
            state_dates, state_values,
            nat_dates, nat_values,
            metric_label, town_name, state_code, zip_code
        )
        # fig.show()
        save_plot(fig, f"{graphs_dir}/{zip_code}_{metric_name}_comparison.png")
        
        table_data["US (Observed)"] = (nat_dates, nat_values)

    # --- Create and save yearly summary table ---
    yearly_table = create_yearly_summary(table_data, metric_name)
    display(yearly_table)
    save_table(yearly_table, f"{tables_dir}/{zip_code}_{metric_name}_yearly.csv")
    
    print()

## Section 4: Data Loading

In [44]:
# Load all data
df_zip = load_zip_data(ZIP_DATA_PATH, ZIP_CODE)
df_state = load_state_data(STATE_DATA_PATH, STATE_CODE)
df_national = load_national_data(NATIONAL_DATA_PATH)
town_name = get_town_name(df_zip).capitalize()


print(f"Data loaded successfully!")
print(f"  Location: {town_name} ({ZIP_CODE})")
print(f"  State: {STATE_CODE}")
print(f"  ZIP data range: {df_zip.index.min()} to {df_zip.index.max()}")
print(f"  Records: {len(df_zip)} months")

Data loaded successfully!
  Location: Waterville (04901)
  State: ME
  ZIP data range: 2016-07-01 00:00:00 to 2025-12-01 00:00:00
  Records: 114 months


## Section 5: Run Analysis

In [45]:
# Metric display names
METRIC_LABELS = {
    'active_listing_count': 'Active Listing Count',
    'median_listing_price': 'Median Listing Price ($)',
    'average_listing_price': 'Average Listing Price ($)',
    'new_listing_count': 'New Listing Count',
    'median_days_on_market': 'Median Days on Market',
    'median_square_feet': 'Median Square Feet',
    'median_listing_price_per_square_foot': 'Median Price per Sq Ft ($)',
}

# Build config dict
# config = {
#     'include_state': INCLUDE_STATE_COMPARISON,
#     'include_national': INCLUDE_NATIONAL_COMPARISON,
#     'apply_seasonal': APPLY_SEASONAL_ADJUSTMENT,
#     'graphs_dir': GRAPHS_OUTPUT_DIR,
#     'tables_dir': TABLES_OUTPUT_DIR,
#     'zip_code': ZIP_CODE,
# }


# display(Markdown(f"# Housing Analysis: {town_name} ({ZIP_CODE})"))
print(f" Housing Analysis: {town_name} ({ZIP_CODE})")
# print(f"Configuration:")
# print(f"  State comparison: {INCLUDE_STATE_COMPARISON}")
# print(f"  National comparison: {INCLUDE_NATIONAL_COMPARISON}")
# print(f"  Seasonal adjustment: {APPLY_SEASONAL_ADJUSTMENT}")
# print()

# Process all metrics
for metric in METRICS:
    print(f'{metric}: Starting...')
    analyze_metric(
        metric_name=metric,
        metric_label=METRIC_LABELS[metric],
        df_zip=df_zip,
        df_state=df_state,
        df_national=df_national,
        town_name=town_name,
        state_code=STATE_CODE,
        config=METRIC_CONFIG[metric]
    )
    print(f'{metric}: Finished')

print("=" * 60)
print(f"Analysis complete!")
print(f"  Graphs saved to: {GRAPHS_OUTPUT_DIR}/")
print(f"  Tables saved to: {TABLES_OUTPUT_DIR}/")

 Housing Analysis: Waterville (04901)
active_listing_count: Starting...


### Active Listing Count

08:29:13 - cmdstanpy - INFO - Chain [1] start processing
08:29:13 - cmdstanpy - INFO - Chain [1] done processing


  Applying seasonal adjustment...
  Saved: outputs/graphs/04901_active_listing_count_seasonal.png
  Saved: outputs/graphs/04901_active_listing_count_vs_ME.png


Year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Waterville (Observed),1081.0,8351.0,7784.0,7257.0,6174.0,5468.0,5474.0,5303.0,6058.0,6484.0
Waterville (Seasonally Adjusted),1141.0,8242.0,7616.0,6953.0,6272.0,5621.0,5426.0,5642.0,6004.0,6370.0
ME (Observed),2038.0,15950.0,15736.0,15500.0,14748.0,13729.0,13501.0,13573.0,14036.0,14389.0


...

### Median Price per Sq Ft ($)

08:29:43 - cmdstanpy - INFO - Chain [1] start processing
08:29:43 - cmdstanpy - INFO - Chain [1] done processing


  Applying seasonal adjustment...
  Saved: outputs/graphs/04901_median_listing_price_per_square_foot_seasonal.png
  Saved: outputs/graphs/04901_median_listing_price_per_square_foot_vs_ME.png
  Saved: outputs/graphs/04901_median_listing_price_per_square_foot_comparison.png


Year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Waterville (Observed),79.5,82.8,86.6,90.5,100.8,120.0,138.0,165.6,170.8,169.2
Waterville (Seasonally Adjusted),79.2,81.6,86.4,91.5,102.3,120.9,140.6,159.2,168.4,173.3
ME (Observed),131.5,133.4,140.2,149.0,165.2,194.2,224.8,254.5,271.1,284.6
US (Observed),126.2,131.9,141.8,149.8,163.1,192.0,216.7,221.1,227.1,227.8


  Saved: outputs/tables/04901_median_listing_price_per_square_foot_yearly.csv

median_listing_price_per_square_foot: Finished
Analysis complete!
  Graphs saved to: outputs/graphs/
  Tables saved to: outputs/tables/
